# GoMLX Korean Opinion Analyzer - Colab TPU Runner

1. 런타임 유형을 **TPU**로 변경하세요. (`런타임` -> `런타임 유형 변경` -> `TPU`)
2. Google Drive(`MyDrive/gomlx-korean-opinion`)에 빌드된 `opinion-analyzer` 바이너리와 `config/` 폴더가 있으면 그대로 복사해 실행하고, 없으면 GitHub 저장소에서 소스를 받아 Go로 자동 빌드 후 Drive에 캐시합니다.
3. `go-xla`가 Colab의 `libtpu.so`를 인식할 수 있도록 `pjrt_c_api_tpu_plugin.so` 심볼릭 링크와 `PJRT_PLUGIN_LIBRARY_PATH`(디렉터리 경로)를 Python 런타임 환경변수에 설정합니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import subprocess
from pathlib import Path

WORK_DIR = Path(os.environ.get("WORK_DIR", "/content/work"))
PJRT_DIR = WORK_DIR / "pjrt"
PJRT_DIR.mkdir(parents=True, exist_ok=True)

libtpu_path = ""
try:
    import libtpu
    libtpu_path = libtpu.get_library_path()
except Exception:
    res = subprocess.run(
        ["bash", "-lc", "find /usr /lib -name 'libtpu.so' 2>/dev/null | head -n 1"],
        capture_output=True,
        text=True,
    )
    libtpu_path = res.stdout.strip()

print(f"Found libtpu.so: {libtpu_path or 'NOT_FOUND'}")

if libtpu_path and Path(libtpu_path).is_file():
    for link_dir in (PJRT_DIR, Path("/usr/local/lib/go-xla")):
        link_dir.mkdir(parents=True, exist_ok=True)
        link_path = link_dir / "pjrt_c_api_tpu_plugin.so"
        if link_path.is_symlink() or link_path.exists():
            link_path.unlink()
        link_path.symlink_to(libtpu_path)

    # go-xla PJRT_PLUGIN_LIBRARY_PATH expects colon-separated directories containing pjrt_c_api_*_plugin.so
    existing_pjrt = os.environ.get("PJRT_PLUGIN_LIBRARY_PATH", "")
    os.environ["PJRT_PLUGIN_LIBRARY_PATH"] = f"{PJRT_DIR}:{existing_pjrt}" if existing_pjrt else str(PJRT_DIR)

    libtpu_dir = str(Path(libtpu_path).parent)
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = f"{libtpu_dir}:{existing_ld}" if existing_ld else libtpu_dir
    os.environ["GOMLX_NO_AUTO_INSTALL"] = "1"
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = os.environ.get("TF_CPP_MIN_LOG_LEVEL", "3")
    os.environ["GOMLX_BACKEND"] = os.environ.get("GOMLX_BACKEND", "xla:tpu")
    os.environ["LIBTPU_PATH"] = libtpu_path

    print(f"PJRT_PLUGIN_LIBRARY_PATH={os.environ['PJRT_PLUGIN_LIBRARY_PATH']}")
    print(f"GOMLX_BACKEND={os.environ['GOMLX_BACKEND']}")
else:
    os.environ["GOMLX_BACKEND"] = os.environ.get("GOMLX_BACKEND", "xla:cpu")
    print(f"WARNING: libtpu.so not found; fallback GOMLX_BACKEND={os.environ['GOMLX_BACKEND']}")

Found libtpu.so: /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
PJRT_PLUGIN_LIBRARY_PATH=/content/work/pjrt
GOMLX_BACKEND=xla:tpu


In [3]:
%%bash
set -euo pipefail

# Google Drive 및 GitHub 소스 설정 (필요 시 환경변수로 덮어쓰기 가능)
DRIVE_DIR="${DRIVE_DIR:-/content/drive/MyDrive/gomlx-korean-opinion}"
WORK_DIR="${WORK_DIR:-/content/work}"
SRC_DIR="${SRC_DIR:-/content/src/gomlx-korean-opinion}"
REPO_URL="${REPO_URL:-https://github.com/Mireu-Lab/gomlx-korean-opinion.git}"
REPO_BRANCH="${REPO_BRANCH:-feature/ko-hub-multilingual-embedding}"
FORCE_REBUILD="${FORCE_REBUILD:-false}"
CACHE_TO_DRIVE="${CACHE_TO_DRIVE:-true}"

BIN_SRC="${DRIVE_DIR}/opinion-analyzer"
CFG_SRC="${DRIVE_DIR}/config"
BIN_DST="${WORK_DIR}/opinion-analyzer"
CFG_DST="${WORK_DIR}/config"

mkdir -p "${WORK_DIR}"

if [ "${FORCE_REBUILD}" != "true" ] && [ -f "${BIN_SRC}" ] && [ -f "${CFG_SRC}/config.yaml" ]; then
  echo "Using cached binary and config from Google Drive: ${DRIVE_DIR}"
  cp "${BIN_SRC}" "${BIN_DST}"
  rm -rf "${CFG_DST}"
  cp -r "${CFG_SRC}" "${CFG_DST}"
else
  echo "Drive binary/config not found (or FORCE_REBUILD=true). Building from ${REPO_URL} (${REPO_BRANCH})..."
  if [ -d "${SRC_DIR}/.git" ]; then
    git -C "${SRC_DIR}" fetch --depth 1 origin "${REPO_BRANCH}"
    git -C "${SRC_DIR}" checkout -B "${REPO_BRANCH}" "origin/${REPO_BRANCH}"
  else
    rm -rf "${SRC_DIR}"
    mkdir -p "$(dirname "${SRC_DIR}")"
    git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" "${SRC_DIR}"
  fi

  export PATH="/usr/local/go/bin:${PATH}"
  REQ_GO_VER="$(awk '/^go /{print $2; exit}' "${SRC_DIR}/go.mod")"
  REQ_MINOR="$(echo "${REQ_GO_VER}" | awk -F. '{print $1*100 + $2}')"
  CUR_MINOR=0
  if command -v go >/dev/null 2>&1; then
    CUR_GO_VER="$(go env GOVERSION 2>/dev/null | sed 's/^go//')"
    CUR_MINOR="$(echo "${CUR_GO_VER}" | awk -F. '{print $1*100 + $2}')"
  fi

  if [ "${CUR_MINOR}" -lt "${REQ_MINOR}" ]; then
    GO_TAR_VER="$(curl -fsSL 'https://go.dev/VERSION?m=text' | head -n 1 | tr -d '\r[:space:]')"
    echo "Installing ${GO_TAR_VER} (required >= go${REQ_GO_VER})..."
    curl -fsSL "https://go.dev/dl/${GO_TAR_VER}.linux-amd64.tar.gz" -o /tmp/go.tar.gz
    rm -rf /usr/local/go
    tar -C /usr/local -xzf /tmp/go.tar.gz
    rm -f /tmp/go.tar.gz
  fi
  go version

  cd "${SRC_DIR}"
  go build -o "${BIN_DST}" ./cmd/opinion-analyzer
  rm -rf "${CFG_DST}"
  cp -r "${SRC_DIR}/config" "${CFG_DST}"

  if [ "${CACHE_TO_DRIVE}" = "true" ] && [ -d "/content/drive/MyDrive" ]; then
    mkdir -p "${DRIVE_DIR}"
    cp "${BIN_DST}" "${BIN_SRC}"
    rm -rf "${CFG_SRC}"
    cp -r "${CFG_DST}" "${CFG_SRC}"
    echo "Cached built binary and config to ${DRIVE_DIR}"
  fi
fi

chmod +x "${BIN_DST}"
cd "${WORK_DIR}"
./opinion-analyzer 2>&1 || true

Drive binary/config not found (or FORCE_REBUILD=true). Building from https://github.com/Mireu-Lab/gomlx-korean-opinion.git (feature/ko-hub-multilingual-embedding)...
Installing go1.27.4 (required >= go1.27)...
go version go1.27.4 linux/amd64
Cached built binary and config to /content/drive/MyDrive/gomlx-korean-opinion
Usage: opinion-analyzer <command> [flags]

Commands:
  download     Download NSMC and KLUE-YNAT datasets configured in config.yaml
  build-graph     Download Unihan/OMW sources and build KO-Hub multilingual roots CSR graph
  pretrain        Run GoMLX Context-Aware Soft Edge contrastive pretraining for KO-Hub Student
  export-dataset  Export preprocessed Parquet dataset splits, vocab, and KO-Hub graph metadata
  train           Fit Korean tokenizer and train GoMLX ReLU multi-task model
  eval         Evaluate saved checkpoint on KLUE-YNAT val, NSMC test, and NFC
  analyze      Classify and sort news articles by topic, Nuclear Feeling, and intensity
  cluster      Run 3-met

Cloning into '/content/src/gomlx-korean-opinion'...
go: downloading github.com/gomlx/compute v0.1.32
go: downloading github.com/gomlx/gomlx v0.28.38
go: downloading gopkg.in/yaml.v3 v3.0.1
go: downloading github.com/parquet-go/parquet-go v0.32.1
go: downloading github.com/gomlx/onnx-gomlx v0.5.35
go: downloading github.com/gomlx/go-xla v0.4.32
go: downloading github.com/pkg/errors v0.9.1
go: downloading github.com/google/uuid v1.6.0
go: downloading github.com/parquet-go/bitpack v1.0.3
go: downloading github.com/parquet-go/jsonlite v1.5.5
go: downloading github.com/twpayne/go-geom v1.6.1
go: downloading golang.org/x/sys v0.48.0
go: downloading google.golang.org/protobuf v1.36.12
go: downloading k8s.io/klog/v2 v2.140.0
go: downloading golang.org/x/exp v0.0.0-20260908205506-85c1c2202aba
go: downloading github.com/andybalholm/brotli v1.3.0
go: downloading github.com/klauspost/compress v1.20.4
go: downloading github.com/pierrec/lz4/v4 v4.1.33
go: downloading github.com/go-logr/logr v1.4.3
g

In [4]:
%%bash
set -euo pipefail

WORK_DIR="${WORK_DIR:-/content/work}"
DRIVE_DIR="${DRIVE_DIR:-/content/drive/MyDrive/gomlx-korean-opinion}"
COMMAND="${COMMAND:-pipeline}"
BACKEND="${GOMLX_BACKEND:-xla:tpu}"
PROVIDER="${EMBEDDING_PROVIDER:-ko_hub_cnn_relu}"
PRETRAIN_STEPS="${PRETRAIN_STEPS:-600}"
TRAIN_STEPS="${TRAIN_STEPS:-2500}"
PCA_DIM="${PCA_DIM:-24}"
SAVE_TO_DRIVE="${SAVE_TO_DRIVE:-true}"

cd "${WORK_DIR}"
echo "Running '${COMMAND}' on backend='${BACKEND}', provider='${PROVIDER}'"
echo "PJRT_PLUGIN_LIBRARY_PATH='${PJRT_PLUGIN_LIBRARY_PATH:-}'"

if [ "${COMMAND}" = "pipeline" ] && [ "${PROVIDER}" != "scratch_relu" ] && [ "${PROVIDER}" != "onnx_backbone_relu" ]; then
  if [ ! -f "checkpoints/ko_hub_pretrain/pretrain_meta.json" ]; then
    ./opinion-analyzer pretrain \
      --config config/config.yaml \
      --backend "${BACKEND}" \
      --steps "${PRETRAIN_STEPS}"
  fi
fi

./opinion-analyzer "${COMMAND}" \
  --config config/config.yaml \
  --backend "${BACKEND}" \
  --provider "${PROVIDER}" \
  --steps "${TRAIN_STEPS}" \
  --pca-dim "${PCA_DIM}"

if [ "${SAVE_TO_DRIVE}" = "true" ] && [ -d "/content/drive/MyDrive" ]; then
  mkdir -p "${DRIVE_DIR}/outputs" "${DRIVE_DIR}/checkpoints" "${DRIVE_DIR}/logs"
  cp -r outputs/. "${DRIVE_DIR}/outputs/" 2>/dev/null || true
  cp -r checkpoints/. "${DRIVE_DIR}/checkpoints/" 2>/dev/null || true
  cp -r logs/. "${DRIVE_DIR}/logs/" 2>/dev/null || true
  echo "Synced outputs, checkpoints, and logs to ${DRIVE_DIR}"
fi

Running 'pipeline' on backend='xla:tpu', provider='ko_hub_cnn_relu'
PJRT_PLUGIN_LIBRARY_PATH='/content/work/pjrt'
[2026-11-28T00:00:35Z] [INFO] [taxonomy] loaded 7 KLUE-YNAT topics and 16 Nuclear Feeling Checklist categories
[2026-11-28T00:00:35Z] [PRETRAIN] [START] preparing KO-Hub graph & corpus for contrastive pretraining (steps=600, out_dim=128)
[2026-11-28T00:00:35Z] [BUILD_GRAPH] [START] downloading/verifying multilingual roots and building KO-Hub CSR graph
[2026-11-28T00:00:35Z] [INFO] [graphkg.sources] downloading Unicode Unihan from https://www.unicode.org/Public/UCD/latest/ucd/Unihan.zip -> data/multilingual_roots/Unihan.zip
[2026-11-28T00:00:36Z] [INFO] [graphkg.sources] downloading OMW Korean Wikt from https://raw.githubusercontent.com/omwn/omw-data/main/wns/wikt/wn-wikt-kor.tab -> data/multilingual_roots/wn-wikt-kor.tab
[2026-11-28T00:00:36Z] [INFO] [graphkg.sources] downloading OMW Korean CLDR from https://raw.githubusercontent.com/omwn/omw-data/main/wns/cldr/wn-cldr-kor.